# Smoke Test - Agentic Evaluation Framework
Tests all 4 patterns using MockLLMClient (no GPU required).

In [2]:
import sys, os, zipfilefrom google.colab import drive# Mount Google Drive and copy agentic-eval.zipdrive.mount("/content/drive")drive_zip = "/content/drive/MyDrive/agentic-eval.zip"if os.path.exists(drive_zip):    with zipfile.ZipFile(drive_zip, "r") as z:        z.extractall("/content/")    print("Extracted agentic-eval.zip from Drive")elif not os.path.isdir("/content/agentic-eval"):    raise FileNotFoundError(        f"{drive_zip} not found. Upload agentic-eval.zip to the root of your Google Drive."    )# Locate src/ directory and add to pathcandidates = ["/content/agentic-eval", "/content"]for p in candidates:    if os.path.isdir(os.path.join(p, "src")):        sys.path.insert(0, p)        print(f"Found src at: {p}")        breakelse:    raise RuntimeError(f"agentic-eval/src not found. Tried: {candidates}")from src.llm_client import MockLLMClientfrom src.schemas import RunState, PatternIDfrom src.patterns import (    GovernedWorkflow,    RoleSeparatedPipeline,    ReflexiveSingleAgent,    MultiPerspectiveDeliberation,)print("All modules imported successfully.")

Mounted at /content/drive
Extracted agentic-eval.zip from Drive
Found src at: /content/agentic-eval
All modules imported successfully.


In [3]:
client = MockLLMClient()

patterns = [
    (PatternID.GOVERNED_WORKFLOW, GovernedWorkflow(client)),
    (PatternID.ROLE_SEPARATED_PIPELINE, RoleSeparatedPipeline(client)),
    (PatternID.REFLEXIVE_SINGLE_AGENT, ReflexiveSingleAgent(client, max_iterations=2)),
    (PatternID.MULTI_PERSPECTIVE_DELIBERATION, MultiPerspectiveDeliberation(client, num_agents=3)),
]

state = RunState(
    run_id="smoke-001", pattern=PatternID.GOVERNED_WORKFLOW,
    dataset="ifeval", query_id="demo",
    query="Answer in two sentences: what is INESC TEC?",
    reference="INESC TEC is a research institute in Porto.",
)

for pid, orchestrator in patterns:
    s = orchestrator.run(state.model_copy(update={"run_id": f"smoke-{pid}", "pattern": pid}))
    print(f"Pattern {pid}: {s.final_answer[:60]}... ({len(s.calls)} calls, {s.rounds} rounds)")

Pattern 1: [mock:solo:r0]... (1 calls, 1 rounds)
Pattern 2: [mock:analyst:r0]

---
[Reviewer: [mock:reviewer:r0]]... (4 calls, 1 rounds)
Pattern 3: [mock:editor:r1]... (7 calls, 2 rounds)
Pattern 4: [mock:judge:r0]... (11 calls, 2 rounds)


In [4]:
from src.metrics import compute_u_L, compute_u_P, compute_surface
from src.metrics.u_C import compute_u_C  # lazy: downloads all-MiniLM-L6-v2 on first call
from src.metrics.u_R import compute_u_R  # lazy: downloads all-MiniLM-L6-v2 on first call

from src.metrics import compute_u_L, compute_u_P, compute_surfacefrom src.metrics.u_C import compute_u_C  # lazy: downloads all-MiniLM-L6-v2 on first callfrom src.metrics.u_R import compute_u_R  # lazy: downloads all-MiniLM-L6-v2 on first callpid = PatternID.MULTI_PERSPECTIVE_DELIBERATIONstate = RunState(    run_id="metric-test", pattern=pid,    dataset="finqa", query_id="m1",    query="What is revenue?",    reference="Revenue was $10M in 2023.",)result = MultiPerspectiveDeliberation(MockLLMClient(), num_agents=3).run(state)analyst_calls = [c for c in result.calls if c.agent_role.startswith("analyst_")]u_c = compute_u_C([c.response_text for c in analyst_calls])print(f"u_C (inter-agent divergence): {u_c:.4f}")u_l = compute_u_L(result.final_answer or "")print(f"u_L (linguistic stability): {u_l:.4f}")u_p = compute_u_P(None, 0.0)print(f"u_P (plausibility paradox): {u_p:.4f}")surface = compute_surface(u_l, None, u_p)print(f"U_surface: {surface:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

u_C (inter-agent divergence): 0.0236
u_L (linguistic stability): 0.0000
u_P (plausibility paradox): 0.0000
U_surface: 0.0000
